# LASTDANCE — SigLIP production (9 batch)

Notebook này chỉ build **SigLIP** trên Kaggle T4 của tài khoản thứ hai. CLIP đang chạy ở notebook/tài khoản khác và không bị sửa hay chờ.

Remote artifact chỉ được phép nằm dưới `siglip/archives/{batch_id}/`; `clip/...` là read/write-protected bởi guard fail-closed trong runner.


## Chuẩn bị trên Kaggle

- Accelerator: **GPU T4**.
- Internet: **On**.
- Gắn đúng dataset keyframe chứa `frames.csv`, `frames.csv.state.json` và các thư mục batch; runner sẽ dùng `AIC_DATA` hoặc tự dò mount Kaggle.
- Tạo Kaggle Secret tên `HF_TOKEN` bằng token của tài khoản bạn, có quyền Write vào private Dataset `MinhThuw0103/lastdance-visual-embeddings`.
- Không ghi hoặc in token trực tiếp trong notebook.


## 1. Clone và khóa đúng commit


In [ ]:
from pathlib import Path
import os
import socket
import subprocess
import sys

WORKING_ROOT = Path("/kaggle/working")
REPO = WORKING_ROOT / "LASTDANCE"
BRANCH = "codex/offline-visual-embeddings"
EXPECTED_COMMIT = "31e7d02294bb219ef85694fb62c646fd02c51538"
CLONE_URL = "https://" + "github.com/ThanhVu165/LASTDANCE.git"

assert not any(marker in CLONE_URL for marker in "[]()"), CLONE_URL

try:
    socket.getaddrinfo("github.com", 443)
except socket.gaierror as error:
    raise RuntimeError(
        "Không phân giải được github.com. Hãy bật Internet trong "
        "Kaggle Notebook Settings rồi khởi động lại session."
    ) from error

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(
        f"{REPO} tồn tại nhưng không phải Git repo. "
        "Đây có thể là clone dở dang; hãy khởi động session Kaggle mới."
    )

if not (REPO / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            CLONE_URL,
            str(REPO),
        ],
        check=True,
    )

subprocess.run(
    ["git", "fetch", "origin", BRANCH],
    cwd=REPO,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", EXPECTED_COMMIT],
    cwd=REPO,
    check=True,
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("CODE CHECKOUT PASS:", actual_commit)


## 2. Cài dependency Kaggle GPU


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/LASTDANCE")
assert (REPO / "offline").is_dir(), REPO
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements/kaggle-gpu.txt",
    ],
    cwd="/kaggle/working/LASTDANCE",
    check=True,
)
print("DEPENDENCY INSTALL PASS")


## 3. Kiểm tra T4, revision và dimension SigLIP


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/LASTDANCE")
assert (REPO / "offline").is_dir(), REPO

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN and HF_TOKEN.strip(), "Thiếu Kaggle Secret HF_TOKEN"

subprocess_env = os.environ.copy()
subprocess_env["HF_TOKEN"] = HF_TOKEN

subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.environment_doctor",
        "--profile",
        "kaggle-gpu",
        "--skip-data",
    ],
    cwd=REPO,
    check=True,
    env=subprocess_env,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.verify_visual_model_revisions",
    ],
    cwd=REPO,
    check=True,
    env=subprocess_env,
)

model_preflight_code = 'import json\nimport sys\nfrom pathlib import Path\n\nREPO = Path("/kaggle/working/LASTDANCE")\nsys.path.insert(0, str(REPO))\n\nimport torch\nfrom transformers import AutoConfig\nfrom offline.visual_models import load_model_config\n\nassert torch.cuda.is_available(), "CUDA unavailable"\ngpu_name = torch.cuda.get_device_name(0)\nassert gpu_name == "Tesla T4", gpu_name\n\nmodel = load_model_config("siglip")\nconfig = AutoConfig.from_pretrained(\n    model["model_id"],\n    revision=model["revision"],\n)\nassert config.model_type == "siglip", config.model_type\nresolved_dim = int(config.vision_config.hidden_size)\nassert resolved_dim == 768, resolved_dim\n\nprint(json.dumps({\n    "gpu_name": gpu_name,\n    "model_id": model["model_id"],\n    "revision": model["revision"],\n    "vector_dim": resolved_dim,\n}))\n'
model_preflight = subprocess.run(
    [sys.executable, "-c", model_preflight_code],
    cwd=REPO,
    check=True,
    text=True,
    capture_output=True,
    env=subprocess_env,
)

if model_preflight.stderr.strip():
    print(model_preflight.stderr, file=sys.stderr, end="")
print(model_preflight.stdout, end="")

model_report = json.loads(
    model_preflight.stdout.strip().splitlines()[-1]
)

print("SIGLIP RUNTIME PREFLIGHT PASS")
print("GPU:", model_report["gpu_name"])
print("model_id:", model_report["model_id"])
print("revision:", model_report["revision"])
print("vector_dim:", model_report["vector_dim"])


## 4. Chạy production SigLIP đủ 9 batch

Cell duy nhất dưới đây thực hiện: restore batch đã upload → intentional interrupt cho batch mới → resume bằng process mới → validate → archive → checksum → upload/verify HF. Upload lỗi sẽ dừng trước batch kế tiếp.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
from pathlib import Path, PurePosixPath

from huggingface_hub import (
    CommitOperationAdd,
    HfApi,
    hf_hub_download,
)
from kaggle_secrets import UserSecretsClient


# ============================================================
# Cấu hình cố định
# ============================================================

REPO = Path("/kaggle/working/LASTDANCE")
assert (REPO / "offline").is_dir(), REPO
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
EXPECTED_COMMIT = (
    "31e7d02294bb219ef85694fb62c646fd02c51538"
)

AIC_DATA_VALUE = os.environ.get("AIC_DATA", "").strip()

if AIC_DATA_VALUE:
    INPUT_ROOT = Path(AIC_DATA_VALUE).expanduser().resolve()
else:
    KAGGLE_INPUT_ROOT = Path("/kaggle/input")
    catalog_candidates = set()

    for pattern in (
        "*/frames.csv",
        "*/*/frames.csv",
        "*/*/*/frames.csv",
        "*/*/*/*/frames.csv",
    ):
        for catalog_candidate in KAGGLE_INPUT_ROOT.glob(pattern):
            candidate_root = catalog_candidate.parent.resolve()
            if (candidate_root / "frames.csv.state.json").is_file():
                catalog_candidates.add(candidate_root)

    assert len(catalog_candidates) == 1, (
        "Hãy gắn đúng keyframe Dataset hoặc đặt AIC_DATA; candidates=",
        sorted(str(path) for path in catalog_candidates),
    )
    INPUT_ROOT = next(iter(catalog_candidates))
    os.environ["AIC_DATA"] = str(INPUT_ROOT)

print("AIC_DATA:", INPUT_ROOT)
CATALOG = INPUT_ROOT / "frames.csv"
CATALOG_STATE = INPUT_ROOT / "frames.csv.state.json"

WORKING_ROOT = Path("/kaggle/working")
WORKER_ROOT = WORKING_ROOT / "production-workers"
MAPPING_REPORT = (
    WORKER_ROOT / "production-batch-mapping.json"
)

EMBEDDING_ROOT = WORKING_ROOT / "visual-embeddings"
ARCHIVE_ROOT = WORKING_ROOT / "siglip-production-archives"
LOG_ROOT = WORKING_ROOT / "siglip-production-logs"
METADATA_ROOT = WORKING_ROOT / "siglip-production-metadata"

HF_REPO_ID = (
    "MinhThuw0103/lastdance-visual-embeddings"
)
HF_REPO_TYPE = "dataset"

BATCH_SIZE = 64
STOP_AFTER_SHARDS = 2

WORKER_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# Hugging Face authentication
# ============================================================

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN and HF_TOKEN.strip(), (
    "Thiếu Kaggle Secret HF_TOKEN"
)

hf_api = HfApi(token=HF_TOKEN)

hf_api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    private=True,
    exist_ok=True,
)

hf_repo_info = hf_api.repo_info(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
)

assert hf_repo_info.private is True, (
    f"Fail closed: {HF_REPO_ID} không phải private Dataset"
)

print("HF AUTH PASS")
print("repo:", HF_REPO_ID)
print("private:", hf_repo_info.private)


# ============================================================
# Fail-closed model/dimension preflight
# ============================================================

SIGLIP_EXPECTED_VECTOR_DIM = 768

model_preflight_env = os.environ.copy()
model_preflight_env["HF_TOKEN"] = HF_TOKEN
model_preflight_code = 'import json\nimport sys\nfrom pathlib import Path\n\nREPO = Path("/kaggle/working/LASTDANCE")\nsys.path.insert(0, str(REPO))\n\nimport torch\nfrom transformers import AutoConfig\nfrom offline.visual_models import load_model_config\n\nassert torch.cuda.is_available(), "CUDA unavailable"\ngpu_name = torch.cuda.get_device_name(0)\nassert gpu_name == "Tesla T4", gpu_name\n\nmodel = load_model_config("siglip")\nconfig = AutoConfig.from_pretrained(\n    model["model_id"],\n    revision=model["revision"],\n)\nassert config.model_type == "siglip", config.model_type\nresolved_dim = int(config.vision_config.hidden_size)\nassert resolved_dim == 768, resolved_dim\n\nprint(json.dumps({\n    "gpu_name": gpu_name,\n    "model_id": model["model_id"],\n    "revision": model["revision"],\n    "vector_dim": resolved_dim,\n}))\n'

model_preflight = subprocess.run(
    [sys.executable, "-c", model_preflight_code],
    cwd=REPO,
    check=True,
    text=True,
    capture_output=True,
    env=model_preflight_env,
)

if model_preflight.stderr.strip():
    print(model_preflight.stderr, file=sys.stderr, end="")
print(model_preflight.stdout, end="")

model_report = json.loads(
    model_preflight.stdout.strip().splitlines()[-1]
)

SIGLIP_MODEL_ROW = {
    "model_id": model_report["model_id"],
    "revision": model_report["revision"],
}

assert model_report["vector_dim"] == SIGLIP_EXPECTED_VECTOR_DIM
assert model_report["gpu_name"] == "Tesla T4"

print("SIGLIP MODEL PREFLIGHT PASS")
print("model_id:", SIGLIP_MODEL_ROW["model_id"])
print("revision:", SIGLIP_MODEL_ROW["revision"])
print("vector_dim:", model_report["vector_dim"])


# ============================================================
# Control files
# ============================================================

CONTROL_FILES = [
    (
        WORKER_ROOT / "production-batch-mapping.json",
        "production-workers/production-batch-mapping.json",
    ),
    *[
        (
            WORKER_ROOT
            / f"embedding-batch-{index:02d}.txt",
            "production-workers/"
            f"embedding-batch-{index:02d}.txt",
        )
        for index in range(1, 10)
    ],
]


def remote_repo_files() -> set[str]:
    return set(
        hf_api.list_repo_files(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
        )
    )


def restore_missing_control_files() -> None:
    missing_local = [
        (local_path, remote_path)
        for local_path, remote_path in CONTROL_FILES
        if not local_path.is_file()
    ]

    if not missing_local:
        return

    repo_files = remote_repo_files()

    missing_remote = [
        remote_path
        for _, remote_path in missing_local
        if remote_path not in repo_files
    ]

    assert not missing_remote, (
        "Không thể restore production control files từ HF:",
        missing_remote,
    )

    for local_path, remote_path in missing_local:
        downloaded = Path(
            hf_hub_download(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                filename=remote_path,
                token=HF_TOKEN,
            )
        )

        local_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        shutil.copy2(downloaded, local_path)

        print("RESTORED CONTROL:", local_path)


restore_missing_control_files()

for required_path in (
    REPO,
    CATALOG,
    CATALOG_STATE,
    MAPPING_REPORT,
):
    assert required_path.exists(), required_path


# ============================================================
# Helper chung
# ============================================================

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as source:
        for chunk in iter(
            lambda: source.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def checksum_value(path: Path) -> str:
    value = (
        path.read_text(encoding="utf-8")
        .strip()
        .split()[0]
        .lower()
    )

    assert len(value) == 64, path
    assert all(
        character in "0123456789abcdef"
        for character in value
    ), path

    return value


def tail_text(path: Path, lines: int = 15) -> str:
    if not path.is_file():
        return ""

    values = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(values[-lines:])


def run_monitored(
    command: list[str],
    *,
    batch_id: str,
    phase: str,
    artifact_dir: Path,
    expected_returncode: int,
) -> None:
    log_path = (
        LOG_ROOT / f"{batch_id}-siglip-{phase}.log"
    )

    print(f"\n[{batch_id}] START {phase}")
    print("log:", log_path)

    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

        last_report_time = 0.0

        while process.poll() is None:
            time.sleep(20)

            now = time.monotonic()

            if now - last_report_time < 60:
                continue

            checkpoint_path = (
                artifact_dir / "checkpoint.json"
            )

            if checkpoint_path.is_file():
                try:
                    checkpoint = json.loads(
                        checkpoint_path.read_text(
                            encoding="utf-8"
                        )
                    )

                    print(
                        f"[{batch_id}] {phase}:",
                        f"{checkpoint.get('next_index', 0)}/"
                        f"{checkpoint.get('total', '?')}",
                        "shards=",
                        checkpoint.get("completed_shards"),
                    )

                except (
                    OSError,
                    json.JSONDecodeError,
                ):
                    pass

            last_report_time = now

        returncode = process.wait()

    log_tail = tail_text(log_path)

    if log_tail:
        print(log_tail)

    if returncode != expected_returncode:
        raise RuntimeError(
            f"{batch_id} {phase} returned "
            f"{returncode}; expected "
            f"{expected_returncode}; "
            f"log={log_path}"
        )

    print(f"[{batch_id}] {phase} PASS")


def write_checksum(path: Path) -> str:
    checksum = sha256_file(path)
    checksum_path = path.with_suffix(
        path.suffix + ".sha256"
    )

    if checksum_path.exists():
        existing = checksum_value(checksum_path)

        assert existing == checksum, (
            checksum_path,
            existing,
            checksum,
        )

    else:
        checksum_path.write_text(
            f"{checksum}  {path.name}\n",
            encoding="utf-8",
        )

    return checksum


# ============================================================
# Archive
# ============================================================

def archive_completed_batch(
    *,
    batch_id: str,
    artifact_dir: Path,
    worker_file: Path,
    manifest: dict,
) -> tuple[Path, str]:
    archive_path = (
        ARCHIVE_ROOT
        / f"lastdance-production-{batch_id}-siglip.tar.gz"
    )

    if archive_path.exists():
        checksum = write_checksum(archive_path)

        print(
            f"[{batch_id}] archive đã tồn tại "
            "và checksum PASS"
        )

        return archive_path, checksum

    handoff_path = (
        METADATA_ROOT / f"{batch_id}-siglip.json"
    )

    handoff = {
        "schema_version": 1,
        "complete": True,
        "batch_id": batch_id,
        "modality": "siglip",
        "code_commit": EXPECTED_COMMIT,
        "record_count": manifest["record_count"],
        "vector_dim": manifest["vector_dim"],
        "vector_dtype": manifest["vector_dtype"],
        "checkpoint_resume_verified": manifest[
            "checkpoint_resume_verified"
        ],
        "runtime": manifest["runtime"],
        "catalog_sha256": sha256_file(CATALOG),
        "worker_sha256": sha256_file(worker_file),
        "overall_publishing_ready": False,
        "publishing_blockers": [
            "clip production is tracked independently and may not be complete",
            "eva_clip dev gate pending; beit3 permanently retired",
            "FAISS indexes are built later on local CPU",
        ],
    }

    handoff_path.write_text(
        json.dumps(
            handoff,
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

    print(
        f"[{batch_id}] creating archive:",
        archive_path,
    )

    with tarfile.open(
        archive_path,
        "w:gz",
    ) as archive:
        archive.add(
            artifact_dir,
            arcname=(
                f"visual-embeddings/{batch_id}/siglip"
            ),
        )
        archive.add(
            worker_file,
            arcname=f"workers/{worker_file.name}",
        )
        archive.add(
            handoff_path,
            arcname=(
                f"metadata/{batch_id}-siglip.json"
            ),
        )

    with tarfile.open(
        archive_path,
        "r:gz",
    ) as archive:
        members = archive.getmembers()
        names = {member.name for member in members}

        for member in members:
            member_path = PurePosixPath(member.name)

            assert not member_path.is_absolute(), (
                member.name
            )
            assert ".." not in member_path.parts, (
                member.name
            )
            assert not member.issym(), member.name
            assert not member.islnk(), member.name

        required_names = {
            (
                f"visual-embeddings/{batch_id}/"
                "siglip/manifest.json"
            ),
            (
                f"visual-embeddings/{batch_id}/"
                "siglip/checkpoint.json"
            ),
            f"workers/{worker_file.name}",
            f"metadata/{batch_id}-siglip.json",
        }

        assert required_names <= names, sorted(
            required_names - names
        )

        media_files = [
            name
            for name in names
            if PurePosixPath(name).suffix.lower()
            in {
                ".jpg",
                ".jpeg",
                ".png",
                ".mp4",
            }
        ]

        assert not media_files, media_files[:10]

    checksum = write_checksum(archive_path)

    print(f"[{batch_id}] ARCHIVE PASS")
    print("bytes:", archive_path.stat().st_size)
    print("sha256:", checksum)

    return archive_path, checksum


# ============================================================
# HF archive paths
# ============================================================

def remote_batch_paths(
    batch_id: str,
) -> tuple[str, str]:
    archive_name = (
        f"lastdance-production-{batch_id}-siglip.tar.gz"
    )
    remote_root = f"siglip/archives/{batch_id}"

    # Hard isolation from the concurrently running CLIP publisher.
    assert remote_root.startswith("siglip/archives/")
    assert not remote_root.startswith("clip/")

    return (
        f"{remote_root}/{archive_name}",
        f"{remote_root}/{archive_name}.sha256",
    )


# ============================================================
# Restore completed batch từ HF
# ============================================================

def restore_siglip_batch_from_hf(
    *,
    batch_id: str,
    artifact_dir: Path,
    expected_total: int,
) -> str | None:
    remote_archive, remote_checksum = (
        remote_batch_paths(batch_id)
    )

    repo_files = remote_repo_files()

    has_archive = remote_archive in repo_files
    has_checksum = remote_checksum in repo_files

    if has_archive != has_checksum:
        raise RuntimeError(
            f"{batch_id}: remote dở dang; "
            "archive/checksum không xuất hiện cùng nhau"
        )

    if not has_archive:
        return None

    cached_checksum = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=remote_checksum,
            token=HF_TOKEN,
        )
    )
    expected_sha = checksum_value(cached_checksum)

    local_archive = (
        ARCHIVE_ROOT
        / f"lastdance-production-{batch_id}-siglip.tar.gz"
    )
    local_checksum = local_archive.with_suffix(
        local_archive.suffix + ".sha256"
    )

    if local_archive.is_file():
        actual_sha = sha256_file(local_archive)

        assert actual_sha == expected_sha, (
            f"{batch_id}: archive local khác remote; "
            f"local={actual_sha}, "
            f"remote={expected_sha}"
        )

    else:
        cached_archive = Path(
            hf_hub_download(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                filename=remote_archive,
                token=HF_TOKEN,
            )
        )

        actual_sha = sha256_file(cached_archive)

        assert actual_sha == expected_sha, (
            f"{batch_id}: archive tải từ HF "
            "sai checksum"
        )

        shutil.copy2(
            cached_archive,
            local_archive,
        )

    local_checksum.write_text(
        f"{expected_sha}  {local_archive.name}\n",
        encoding="utf-8",
    )

    if artifact_dir.exists():
        manifest_path = artifact_dir / "manifest.json"

        if not manifest_path.is_file():
            raise RuntimeError(
                f"{batch_id}: HF đã có batch hoàn chỉnh "
                "nhưng artifact local đang partial; "
                "không tự ghi đè"
            )

        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

        assert manifest["complete"] is True
        assert manifest["modality"] == "siglip"
        assert manifest["record_count"] == expected_total

        print(
            f"[{batch_id}] remote archive found; "
            "completed artifact already exists"
        )

        return expected_sha

    print(
        f"[{batch_id}] restoring completed artifact "
        "from HF"
    )

    with tempfile.TemporaryDirectory(
        prefix=f"restore-{batch_id}-",
        dir=str(WORKING_ROOT),
    ) as temporary_directory:
        temporary_root = Path(temporary_directory)

        with tarfile.open(
            local_archive,
            "r:gz",
        ) as archive:
            members = archive.getmembers()

            for member in members:
                member_path = PurePosixPath(
                    member.name
                )

                assert (
                    not member_path.is_absolute()
                ), member.name
                assert (
                    ".." not in member_path.parts
                ), member.name
                assert not member.issym(), member.name
                assert not member.islnk(), member.name

            archive.extractall(
                temporary_root,
                filter="data",
            )

        expected_artifact = (
            temporary_root
            / "visual-embeddings"
            / batch_id
            / "siglip"
        )

        if expected_artifact.is_dir():
            restored_artifact = expected_artifact

        else:
            candidates = []

            for candidate_manifest in (
                temporary_root.rglob("manifest.json")
            ):
                candidate_dir = (
                    candidate_manifest.parent
                )
                candidate_checkpoint = (
                    candidate_dir / "checkpoint.json"
                )

                if not candidate_checkpoint.is_file():
                    continue

                try:
                    candidate_data = json.loads(
                        candidate_manifest.read_text(
                            encoding="utf-8"
                        )
                    )
                except json.JSONDecodeError:
                    continue

                if (
                    candidate_data.get("complete") is True
                    and candidate_data.get("modality")
                    == "siglip"
                    and candidate_data.get("record_count")
                    == expected_total
                ):
                    candidates.append(candidate_dir)

            assert len(candidates) == 1, candidates
            restored_artifact = candidates[0]

        artifact_dir.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copytree(
            restored_artifact,
            artifact_dir,
        )

    restored_manifest = json.loads(
        (artifact_dir / "manifest.json").read_text(
            encoding="utf-8"
        )
    )

    assert restored_manifest["complete"] is True
    assert restored_manifest["modality"] == "siglip"
    assert (
        restored_manifest["record_count"]
        == expected_total
    )

    print(f"[{batch_id}] HF RESTORE PASS")

    return expected_sha


# ============================================================
# Upload completed batch lên HF
# ============================================================

def persist_siglip_archive(
    *,
    batch_id: str,
    archive_path: Path,
) -> str:
    checksum_path = archive_path.with_suffix(
        archive_path.suffix + ".sha256"
    )

    assert archive_path.is_file(), archive_path
    assert checksum_path.is_file(), checksum_path

    actual_sha = sha256_file(archive_path)
    declared_sha = checksum_value(checksum_path)

    assert actual_sha == declared_sha, (
        f"{batch_id}: checksum local không khớp"
    )

    remote_archive, remote_checksum = (
        remote_batch_paths(batch_id)
    )

    repo_files = remote_repo_files()

    has_archive = remote_archive in repo_files
    has_checksum = remote_checksum in repo_files

    if has_archive != has_checksum:
        raise RuntimeError(
            f"{batch_id}: remote dở dang; "
            "không tự ghi đè"
        )

    if has_archive:
        downloaded_checksum = Path(
            hf_hub_download(
                repo_id=HF_REPO_ID,
                repo_type=HF_REPO_TYPE,
                filename=remote_checksum,
                token=HF_TOKEN,
                force_download=True,
            )
        )

        remote_sha = checksum_value(
            downloaded_checksum
        )

        assert remote_sha == actual_sha, (
            f"{batch_id}: remote khác local; "
            f"remote={remote_sha}, "
            f"local={actual_sha}"
        )

        print(
            f"[{batch_id}] HF SKIP: "
            "archive đã tồn tại đúng checksum"
        )

    else:
        commit_info = hf_api.create_commit(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            operations=[
                CommitOperationAdd(
                    path_in_repo=remote_archive,
                    path_or_fileobj=str(
                        archive_path
                    ),
                ),
                CommitOperationAdd(
                    path_in_repo=remote_checksum,
                    path_or_fileobj=str(
                        checksum_path
                    ),
                ),
            ],
            commit_message=(
                f"data(siglip): publish production "
                f"{batch_id}"
            ),
        )

        print(
            f"[{batch_id}] HF UPLOAD PASS:",
            commit_info.oid,
        )

    verified_checksum = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=remote_checksum,
            token=HF_TOKEN,
            force_download=True,
        )
    )

    assert (
        checksum_value(verified_checksum)
        == actual_sha
    )

    remote_commit = hf_api.repo_info(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
    ).sha

    print(f"[{batch_id}] HF REMOTE VERIFY PASS")
    print("repo:", HF_REPO_ID)
    print("commit:", remote_commit)
    print("sha256:", actual_sha)

    return remote_commit


# ============================================================
# Preflight code commit + mapping
# ============================================================

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

assert commit == EXPECTED_COMMIT, (
    commit,
    EXPECTED_COMMIT,
)

mapping = json.loads(
    MAPPING_REPORT.read_text(encoding="utf-8")
)

assert mapping["complete"] is True
assert mapping["batch_count"] == 9
assert mapping["video_count"] == 873
assert mapping["keyframe_count"] == 293336
assert (
    mapping["catalog_sha256"]
    == sha256_file(CATALOG)
)

batch_rows = mapping["batches"]

assert [
    row["batch_id"]
    for row in batch_rows
] == [
    f"batch-{number:02d}"
    for number in range(1, 10)
]

seen_video_ids = set()

for row in batch_rows:
    batch_id = row["batch_id"]
    keyframes_root = (
        INPUT_ROOT / row["keyframes_batch"]
    )
    worker_file = (
        WORKER_ROOT / row["worker_file"]
    )

    assert keyframes_root.is_dir(), keyframes_root
    assert worker_file.is_file(), worker_file

    video_ids = [
        line.strip()
        for line in worker_file.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    assert len(video_ids) == row["video_count"]
    assert len(video_ids) == len(set(video_ids))
    assert not (
        seen_video_ids & set(video_ids)
    ), batch_id

    seen_video_ids.update(video_ids)

assert len(seen_video_ids) == 873

print("ALL-BATCH PREFLIGHT PASS")
print("commit:", commit)
print("batches:", len(batch_rows))
print("videos:", len(seen_video_ids))
print("keyframes:", mapping["keyframe_count"])


# ============================================================
# Xác minh control files trên HF giống local
# ============================================================

repo_files = remote_repo_files()

for local_path, remote_path in CONTROL_FILES:
    assert remote_path in repo_files, (
        f"HF thiếu control file: {remote_path}"
    )

    downloaded = Path(
        hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=remote_path,
            token=HF_TOKEN,
        )
    )

    assert (
        sha256_file(downloaded)
        == sha256_file(local_path)
    ), (
        f"Control file remote khác local: "
        f"{remote_path}"
    )

print("HF CONTROL FILES PASS")


# ============================================================
# Lưu catalog/mapping local một lần
# ============================================================

catalog_bundle = (
    ARCHIVE_ROOT
    / "lastdance-production-catalog-mapping.tar.gz"
)

if not catalog_bundle.exists():
    commit_file = (
        METADATA_ROOT / "code-commit.txt"
    )
    freeze_file = (
        METADATA_ROOT / "pip-freeze.txt"
    )

    commit_file.write_text(
        commit + "\n",
        encoding="utf-8",
    )

    freeze_file.write_text(
        subprocess.check_output(
            [
                sys.executable,
                "-m",
                "pip",
                "freeze",
            ],
            text=True,
        ),
        encoding="utf-8",
    )

    with tarfile.open(
        catalog_bundle,
        "w:gz",
    ) as archive:
        archive.add(
            CATALOG,
            arcname="catalog/frames.csv",
        )
        archive.add(
            CATALOG_STATE,
            arcname="catalog/frames.csv.state.json",
        )
        archive.add(
            MAPPING_REPORT,
            arcname=(
                "workers/"
                "production-batch-mapping.json"
            ),
        )

        for row in batch_rows:
            worker_file = (
                WORKER_ROOT / row["worker_file"]
            )
            archive.add(
                worker_file,
                arcname=(
                    f"workers/{worker_file.name}"
                ),
            )

        archive.add(
            commit_file,
            arcname="metadata/code-commit.txt",
        )
        archive.add(
            freeze_file,
            arcname="metadata/pip-freeze.txt",
        )

catalog_bundle_sha = write_checksum(
    catalog_bundle
)

print("CATALOG BUNDLE PASS")
print("archive:", catalog_bundle)
print("sha256:", catalog_bundle_sha)


# ============================================================
# Build → validate → archive → upload toàn bộ 9 batch
# ============================================================

completed_rows = []

for row in batch_rows:
    batch_id = row["batch_id"]
    expected_total = row["keyframe_count"]

    keyframes_root = (
        INPUT_ROOT / row["keyframes_batch"]
    )
    worker_file = (
        WORKER_ROOT / row["worker_file"]
    )
    artifact_dir = (
        EMBEDDING_ROOT / batch_id / "siglip"
    )

    manifest_path = (
        artifact_dir / "manifest.json"
    )
    checkpoint_path = (
        artifact_dir / "checkpoint.json"
    )

    print("\n" + "=" * 72)
    print(
        batch_id,
        "videos=",
        row["video_count"],
        "keyframes=",
        expected_total,
    )

    restored_remote_sha = (
        restore_siglip_batch_from_hf(
            batch_id=batch_id,
            artifact_dir=artifact_dir,
            expected_total=expected_total,
        )
    )

    if restored_remote_sha is not None:
        print(
            f"[{batch_id}] remote persisted:",
            restored_remote_sha,
        )

    base_command = [
        sys.executable,
        "-m",
        "scripts.build_visual_embeddings",
        "--modality",
        "siglip",
        "--batch-id",
        batch_id,
        "--catalog",
        str(CATALOG),
        "--keyframes-root",
        str(keyframes_root),
        "--embedding-root",
        str(EMBEDDING_ROOT),
        "--video-id-file",
        str(worker_file),
        "--batch-size",
        str(BATCH_SIZE),
    ]

    if manifest_path.is_file():
        print(
            f"[{batch_id}] completed artifact "
            "found; validating"
        )

    else:
        if not artifact_dir.exists():
            run_monitored(
                base_command
                + [
                    "--stop-after-shards",
                    str(STOP_AFTER_SHARDS),
                ],
                batch_id=batch_id,
                phase="intentional-stop",
                artifact_dir=artifact_dir,
                expected_returncode=75,
            )

            checkpoint = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            assert checkpoint["complete"] is False
            assert checkpoint["next_index"] == (
                BATCH_SIZE
                * STOP_AFTER_SHARDS
            )
            assert (
                checkpoint["total"]
                == expected_total
            )
            assert (
                checkpoint["completed_shards"]
                == STOP_AFTER_SHARDS
            )
            assert checkpoint[
                "intentional_interruption_observed"
            ] is True
            assert not manifest_path.exists()

        else:
            assert checkpoint_path.is_file(), (
                f"{batch_id}: artifact partial "
                "nhưng thiếu checkpoint"
            )

            checkpoint = json.loads(
                checkpoint_path.read_text(
                    encoding="utf-8"
                )
            )

            assert checkpoint["complete"] is False
            assert (
                checkpoint["total"]
                == expected_total
            )
            assert checkpoint[
                "intentional_interruption_observed"
            ] is True, (
                f"{batch_id}: checkpoint không có "
                "intentional interruption marker; "
                "không tự tiếp tục"
            )

            print(
                f"[{batch_id}] partial checkpoint:",
                f"{checkpoint['next_index']}/"
                f"{expected_total}",
            )

        run_monitored(
            base_command,
            batch_id=batch_id,
            phase="resume",
            artifact_dir=artifact_dir,
            expected_returncode=0,
        )

    validator_command = [
        sys.executable,
        "-m",
        "scripts.validate_visual_embeddings",
        "--artifact-dir",
        str(artifact_dir),
        "--catalog",
        str(CATALOG),
        "--keyframes-root",
        str(keyframes_root),
        "--require-resume-verified",
    ]

    run_monitored(
        validator_command,
        batch_id=batch_id,
        phase="validate",
        artifact_dir=artifact_dir,
        expected_returncode=0,
    )

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )
    checkpoint = json.loads(
        checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    assert manifest["complete"] is True
    assert manifest["modality"] == "siglip"
    assert manifest["model"]["id"] == SIGLIP_MODEL_ROW["model_id"]
    assert manifest["model"]["revision"] == SIGLIP_MODEL_ROW["revision"]
    assert (
        manifest["record_count"]
        == expected_total
    )
    assert manifest["vector_dim"] == SIGLIP_EXPECTED_VECTOR_DIM
    assert manifest["vector_dtype"] == "float16"
    assert manifest[
        "checkpoint_resume_verified"
    ] is True

    assert checkpoint["complete"] is True
    assert (
        checkpoint["next_index"]
        == expected_total
    )
    assert checkpoint[
        "checkpoint_resume_verified"
    ] is True

    runtime = manifest["runtime"]

    assert runtime["device"] == "cuda"
    assert runtime["gpu_name"] == "Tesla T4"

    archive_path, archive_sha = (
        archive_completed_batch(
            batch_id=batch_id,
            artifact_dir=artifact_dir,
            worker_file=worker_file,
            manifest=manifest,
        )
    )

    # Chỉ sang batch tiếp theo sau khi upload và
    # xác minh checksum remote thành công.
    hf_commit = persist_siglip_archive(
        batch_id=batch_id,
        archive_path=archive_path,
    )

    completed_rows.append(
        {
            "batch_id": batch_id,
            "record_count": expected_total,
            "archive": str(archive_path),
            "archive_sha256": archive_sha,
            "hf_repo_id": HF_REPO_ID,
            "hf_repo_type": HF_REPO_TYPE,
            "hf_commit": hf_commit,
            "remote_persisted": True,
            "peak_cuda_memory_bytes": runtime[
                "peak_cuda_memory_bytes"
            ],
        }
    )

    print(
        f"[{batch_id}] "
        "COMPLETE AND REMOTE-PERSISTED"
    )


# ============================================================
# Báo cáo cuối
# ============================================================

assert len(completed_rows) == 9
assert sum(
    row["record_count"]
    for row in completed_rows
) == 293336

assert all(
    row["remote_persisted"]
    for row in completed_rows
)

final_report = {
    "schema_version": 1,
    "complete": True,
    "modality": "siglip",
    "code_commit": commit,
    "batch_count": len(completed_rows),
    "video_count": 873,
    "record_count": sum(
        row["record_count"]
        for row in completed_rows
    ),
    "hf_repo_id": HF_REPO_ID,
    "hf_repo_type": HF_REPO_TYPE,
    "batches": completed_rows,
    "overall_publishing_ready": False,
    "publishing_blockers": [
        "clip production is tracked independently and may not be complete",
        "eva_clip dev gate pending; beit3 permanently retired",
        "FAISS indexes are built later on local CPU",
    ],
}

final_report_path = (
    METADATA_ROOT
    / "siglip-production-report.json"
)

final_report_path.write_text(
    json.dumps(
        final_report,
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

all_archives = sorted(
    ARCHIVE_ROOT.glob("*.tar.gz")
)
checksums = []

for archive_path in all_archives:
    checksum = write_checksum(archive_path)
    checksums.append(
        f"{checksum}  {archive_path.name}"
    )

sums_path = (
    ARCHIVE_ROOT / "SHA256SUMS.txt"
)
sums_path.write_text(
    "\n".join(checksums) + "\n",
    encoding="utf-8",
)

final_remote_commit = hf_api.repo_info(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
).sha

print("\n" + "=" * 72)
print("ALL SIGLIP PRODUCTION BATCHES PASS")
print("batches:", len(completed_rows))
print("videos:", 873)
print("records:", 293336)
print("HF repo:", HF_REPO_ID)
print("HF final commit:", final_remote_commit)
print("report:", final_report_path)
print("archives:", ARCHIVE_ROOT)
print("checksums:", sums_path)